# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns.

We'll enumerate all record sets, their corresponding fields and columns, always referencing by the `@id` of each entity.

In [ ]:
# List all record sets, fields, and columns by `@id`
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}  Name: {rs.name if hasattr(rs, 'name') else '(no name)'}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"  Field @id: {field.id}	 Name: {field.name if hasattr(field, 'name') else '(no name)'}  DataType: {getattr(field, 'data_type', '?')}")
            # Try to print columns' ids
            if hasattr(field, 'columns') and field.columns:
                for column in field.columns:
                    print(f"    Column @id: {column.id}\t Name: {getattr(column, 'name', '(no name)')}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview. We'll extract every record set discovered above.

> **Note:** The `@id` is a unique identifier for each record set. All manipulations will reference these.

In [ ]:
# Extract data from each record set into a DataFrame
# You may need to modify record_set_ids if your dataset lists specific sets to analyze.
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame shape: {dataframes[record_set_id].shape}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head(2))
    else:
        print("No records found or unable to parse data for this RecordSet.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's proceed by selecting a record set containing numeric fields, such as abundance, peptide counts, or molecular weight (MW). **All operations reference fields by their `@id`.**

**Below, set**:
- `selected_record_set_id` to the chosen record set's `@id`,
- `numeric_field_id` to a numeric field's `@id` (e.g., 'cr:field:MW'),
- `group_field_id` to a field to group by (e.g., 'cr:field:Sample' if present).

_Edit the IDs below according to your dataset overview._

In [ ]:
# Example: Set the relevant record set and field @id (edit as appropriate)
# Example placeholder IDs that should be replaced as per real IDs above.

# If you do not know the actual field IDs, run the overview section and copy them here.
selected_record_set_id = record_set_ids[0] if len(record_set_ids) else None  # Example: 'cr:RecordSet:Proteins'
numeric_field_id = None  # Example: 'cr:field:MW'  # Set to the @id of a numeric field (e.g., molecular weight field)
group_field_id = None  # Example: 'cr:field:Sample'  # Set to the @id of a categorical field

# For demonstration, auto-search for first numeric field (float/int) if not set
if selected_record_set_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    # Auto-detect first numeric column if numeric_field_id was not set
    if not numeric_field_id:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                print(f"Auto-selected numeric field: {numeric_field_id}")
                break
    if not group_field_id:
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                print(f"Auto-selected group field: {group_field_id}")
                break

    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'iufc' else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}")
        print(filtered_df.head())

        # Normalize selected numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a field if appropriate and show means
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No valid record set and DataFrame found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll generate a histogram of the chosen numeric field and, if available, a boxplot by groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and selected_record_set_id in dataframes and numeric_field_id:
    df = dataframes[selected_record_set_id]

    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if present
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization could not be created: missing numeric field or DataFrame.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to explore a Croissant-based dataset using the `mlcroissant` library. We loaded all available record sets and fields, performed basic filtering and normalization by referencing fields with their `@id`, and visualized key numeric data. 

- The dataset allows for rich analysis of protein abundance and measurements from extracellular vesicle research.
- By always referencing dataset entities with their `@id`, this process remains robust, machine-actionable, and precise.
- For further work, consider joining multiple record sets, engineering additional features, and applying advanced data science workflows specific to proteomics.